#  EDA — ST1 : Analyse du Funnel de Conversion
**Projet 2 — Segmentation RFM, CLV & Optimisation E-Commerce**


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Setup & Connexion à DuckDB Gold

In [ ]:
# ── Imports ───────────────────────────────────────────────
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Connexion à la base Gold (créée par dbt run) ──────────
GOLD_DB = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb'
con = duckdb.connect(GOLD_DB, read_only=True)

# ── Vérifier les tables disponibles ───────────────────────
tables = con.sql('SHOW TABLES').fetchdf()
print('Tables disponibles dans Gold:')
print(tables)

Tables disponibles dans Gold:
                 name
0        dim_customer
1            dim_date
2         dim_payment
3        fact_returns
4   fact_transactions
5            mart_clv
6        mart_cohorts
7         mart_funnel
8            mart_rfm
9         stg_returns
10       stg_shipping
11   stg_transactions


In [ ]:
# ── Charger les tables utiles pour ST1 ────────────────────

# Sessions converties (mart_funnel = 1 ligne par client x jour)
df_funnel = con.sql('SELECT * FROM mart_funnel').fetchdf()

# Retours clients
df_returns = con.sql('SELECT * FROM fact_returns').fetchdf()

# Shipping (livraison)
df_shipping = con.sql('SELECT * FROM stg_shipping').fetchdf()

print(f'mart_funnel    : {len(df_funnel):,} sessions converties')
print(f'fact_returns   : {len(df_returns):,} retours')
print(f'stg_shipping   : {len(df_shipping):,} commandes shipping')
print()
print('Colonnes mart_funnel :')
print(df_funnel.dtypes)

mart_funnel    : 15,963 sessions converties
fact_returns   : 8,872 retours
stg_shipping   : 10,999 commandes shipping

Colonnes mart_funnel :
customer_id                    object
session_date           datetime64[us]
country                        object
invoice_month                   int32
invoice_week                    int64
basket_value                  float64
nb_invoices                     int64
nb_items                      float64
avg_hour                      float64
weekday                         int32
main_category                  object
dominant_price_tier            object
is_converted                     bool
basket_tier                    object
dtype: object


##Q1:A quelle étape du parcours d'achat les clients abandonnent-ils le plus fréquemment ?


## 1. KPI 1 & 2 — Taux de Conversion Global et Taux d'Abandon

In [ ]:
# ── Hypothèse documentée  ────────────────────────
# Benchmark  : taux conversion e-commerce = 3%
# Funnel : 100% visiteurs → 60% vue produit → 35% ajout panier
#          → 10% checkout → 3% achat finalisé
CONVERSION_RATE = 0.03

# ── Calculs globaux ───────────────────────────────────────
nb_converted   = df_funnel['nb_invoices'].sum()                   # réel
nb_sessions    = round(nb_converted / CONVERSION_RATE)            # estimé
nb_abandoned   = nb_sessions - nb_converted                       # estimé
abandonment_rate = 1 - CONVERSION_RATE

avg_basket_converted = df_funnel['basket_value'].mean()
revenue_gap_total    = nb_abandoned * avg_basket_converted

print('=' * 55)
print(f'  Commandes validées (réel)    : {nb_converted:>10,.0f}')
print(f'  Sessions estimées (benchmark): {nb_sessions:>10,.0f}')
print(f'  Sessions abandonnées (estimé): {nb_abandoned:>10,.0f}')
print(f'  Taux de conversion           : {CONVERSION_RATE*100:>9.1f} %')
print(f'  Taux d\'abandon              : {abandonment_rate*100:>9.1f} %')
print(f'  Panier moyen converti        : {avg_basket_converted:>9.2f} GBP')
print(f'  Manque à gagner total        : {revenue_gap_total:>9,.0f} GBP')
print('=' * 55)

  Commandes validées (réel)    :     17,398
  Sessions estimées (benchmark):    579,933
  Sessions abandonnées (estimé):    562,535
  Taux de conversion           :       3.0 %
  Taux d'abandon              :      97.0 %
  Panier moyen converti        :    350.04 GBP
  Manque à gagner total        : 196,910,854 GBP


In [ ]:
# ── Visualisation : Funnel de conversion (5 étapes) ───────
funnel_data = {
    'Étape': [
        '1. Visiteurs',
        '2. Vue produit',
        '3. Ajout panier',
        '4. Checkout',
        '5. Achat finalisé'
    ],
    'Valeur': [
        nb_sessions,
        round(nb_sessions * 0.60),
        round(nb_sessions * 0.35),
        round(nb_sessions * 0.10),
        nb_converted
    ]
}
df_funnel_chart = pd.DataFrame(funnel_data)
df_funnel_chart['Pct'] = (df_funnel_chart['Valeur'] / nb_sessions * 100).round(1)

fig_funnel = go.Figure(go.Funnel(
    y=df_funnel_chart['Étape'],
    x=df_funnel_chart['Valeur'],
    textinfo='value+percent initial',
    marker_color=['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A']
))
fig_funnel.update_layout(
    title='Funnel de conversion — Observatoire ST1',
    title_font_size=16,
    height=450
)
fig_funnel.show()

##Q2.Certains profils clients (pays, historique d'achat, segment RFM) sont-ils plus enclins a l'abandon ?


## 2. KPI 5 — Taux de Conversion par Pays

In [ ]:
# ── Conversion par pays ────────────────────────────────────
by_country = df_funnel.groupby('country').agg(
    nb_orders     = ('nb_invoices', 'sum'),
    total_revenue = ('basket_value', 'sum'),
    avg_basket    = ('basket_value', 'mean')
).reset_index()

by_country['nb_sessions_est'] = (by_country['nb_orders'] / CONVERSION_RATE).round()
by_country['conversion_rate'] = CONVERSION_RATE * 100
by_country = by_country.sort_values('total_revenue', ascending=False)

print('Top 15 pays par CA :')
print(by_country.head(15)[['country','nb_orders','total_revenue','avg_basket']].to_string(index=False))

Top 15 pays par CA :
        country  nb_orders  total_revenue  avg_basket
 United Kingdom      15653     4745037.55  329.860101
        Germany        438      174530.29  428.821351
         France        369      150068.14  431.230287
           EIRE        249      149937.85  856.787714
    Switzerland         44       41926.90  975.044186
          Spain         84       40076.28  482.846747
    Netherlands         76       39971.60  689.165517
        Belgium         97       33989.82  361.593830
         Norway         32       24931.82  859.717931
       Portugal         49       24920.22  519.171250
      Australia         45       24788.93  604.608049
Channel Islands         22       16134.99  896.388333
          Italy         34       14980.72  440.609412
        Finland         36       13323.68  391.872941
        Denmark         18       12315.88  684.215556


In [ ]:
# ── Carte choroplèthe ──────────────────────────────────────
fig_map = px.choropleth(
    by_country,
    locations='country',
    locationmode='country names',
    color='total_revenue',
    hover_name='country',
    hover_data={'nb_orders': True, 'avg_basket': ':.2f', 'total_revenue': ':.0f'},
    color_continuous_scale='Blues',
    title='Carte de conversion par pays — Taux de conversion & CA'
)
fig_map.update_layout(height=500)
fig_map.show()

# ── Bar chart top pays hors UK ─────────────────────────────
top_nouk = by_country[by_country['country'] != 'United Kingdom'].head(10)
fig_pays = px.bar(
    top_nouk,
    x='total_revenue', y='country',
    orientation='h',
    title='Top 10 pays hors UK — CA total',
    color='avg_basket',
    color_continuous_scale='Viridis',
    labels={'total_revenue': 'CA (GBP)', 'country': 'Pays', 'avg_basket': 'Panier moyen'}
)
fig_pays.update_layout(height=420, yaxis={'categoryorder': 'total ascending'})
fig_pays.show()

##Q3.Existe-t-il des catégories de produits ou des tranches de prix associées a un taux d'abandon structurellement plus élevé ?


## 3. KPI 6 — Taux de Conversion par Catégorie Prix

In [ ]:
# ── Conversion par price_tier ──────────────────────────────
# dominant_price_tier vient de mart_funnel (max price_tier par session)
by_price = df_funnel.groupby('dominant_price_tier').agg(
    nb_sessions_converted = ('nb_invoices', 'sum'),
    total_revenue         = ('basket_value', 'sum'),
    avg_basket            = ('basket_value', 'mean'),
    nb_customers          = ('customer_id', 'nunique')
).reset_index()

by_price['nb_sessions_est']  = (by_price['nb_sessions_converted'] / CONVERSION_RATE).round()
by_price['nb_abandoned']     = by_price['nb_sessions_est'] - by_price['nb_sessions_converted']
by_price['pct_sessions']     = (by_price['nb_sessions_converted'] / by_price['nb_sessions_converted'].sum() * 100).round(1)

print(by_price[['dominant_price_tier','nb_sessions_converted','nb_abandoned',
                'avg_basket','total_revenue','pct_sessions']].to_string(index=False))

dominant_price_tier  nb_sessions_converted  nb_abandoned  avg_basket  total_revenue  pct_sessions
         High (>10)                    118        3815.0  199.858475       23583.30           0.7
       Low (<2 GBP)                    714       23086.0   95.992257       67194.58           4.1
      Medium (2-10)                  16566      535634.0  362.954237     5496941.92          95.2


In [ ]:
# ── Bar chart groupé : sessions converties vs abandonnées ─
fig_prix = go.Figure()
fig_prix.add_trace(go.Bar(
    name='Sessions converties',
    x=by_price['dominant_price_tier'],
    y=by_price['nb_sessions_converted'],
    marker_color='#00CC96'
))
fig_prix.add_trace(go.Bar(
    name='Sessions abandonnées (estimé)',
    x=by_price['dominant_price_tier'],
    y=by_price['nb_abandoned'],
    marker_color='#EF553B'
))
fig_prix.update_layout(
    title='Abandon par catégorie de prix — KPI 6',
    barmode='group',
    xaxis_title='Catégorie prix',
    yaxis_title='Nombre de sessions',
    height=420
)
fig_prix.show()

##Q4.Quel est le manque à gagner mensuel estimé lié aux paniers non convertis?

## 5. KPI 4 — Manque à Gagner Mensuel

In [ ]:
# ── ÉTAPE 1 : Définir le taux de conversion panier → checkout ──
# Benchmark : 35% des visiteurs ajoutent au panier, 10% vont au checkout
# Soit : 10% / 35% = 28,6% des paniers sont convertis en checkout
CART_TO_CHECKOUT_RATE = 0.10 / 0.35  # ≈ 0.2857 (28.6%)

# ── Manque à gagner par mois (PANIER ABANDONNÉ SEULEMENT) ─────
monthly_cart = df_funnel.groupby('invoice_month').agg(
    nb_converted_sessions = ('nb_invoices', 'sum'),           # commandes finalisées
    nb_carts              = ('nb_invoices', 'sum'),
    monthly_revenue       = ('basket_value', 'sum'),
    avg_basket            = ('basket_value', 'mean')
).reset_index()

# Estimer le nombre de paniers créés
# Hypothèse : chaque commande convertie = 1 panier (en réalité, plusieurs paniers peuvent précéder)
# Pour une estimation plus précise : nb_paniers_est = nb_converted_sessions / CART_TO_CHECKOUT_RATE
monthly_cart['nb_carts_est'] = (monthly_cart['nb_converted_sessions'] / CART_TO_CHECKOUT_RATE).round()

# Nombre de paniers abandonnés = paniers créés - commandes finalisées
monthly_cart['nb_cart_abandoned'] = monthly_cart['nb_carts_est'] - monthly_cart['nb_converted_sessions']

# Manque à gagner = paniers abandonnés × panier moyen converti
monthly_cart['revenue_gap_cart'] = monthly_cart['nb_cart_abandoned'] * monthly_cart['avg_basket']
monthly_cart['cumul_gap_cart'] = monthly_cart['revenue_gap_cart'].cumsum()

print("=" * 70)
print("  MANQUE À GAGNER MENSUEL — PANIERS ABANDONNÉS UNIQUEMENT")
print("=" * 70)
print(monthly_cart[['invoice_month', 'nb_converted_sessions', 'nb_cart_abandoned',
                    'avg_basket', 'revenue_gap_cart', 'cumul_gap_cart']].to_string(index=False))



  MANQUE À GAGNER MENSUEL — PANIERS ABANDONNÉS UNIQUEMENT
 invoice_month  nb_converted_sessions  nb_cart_abandoned  avg_basket  revenue_gap_cart  cumul_gap_cart
             1                    935             2337.0  349.895417      8.177056e+05    8.177056e+05
             2                    932             2330.0  329.267565      7.671934e+05    1.584899e+06
             3                   1246             3115.0  336.670938      1.048730e+06    2.633629e+06
             4                   1068             2670.0  319.317347      8.525773e+05    3.486206e+06
             5                   1453             3632.0  344.294030      1.250476e+06    4.736682e+06
             6                   1308             3270.0  322.173099      1.053506e+06    5.790188e+06
             7                   1252             3130.0  325.977611      1.020310e+06    6.810498e+06
             8                   1188             2970.0  349.929464      1.039291e+06    7.849789e+06
             9 

In [ ]:
# ── Visualisation : Manque à gagner sur PANIERS abandonnés ───
fig_gap_cart = make_subplots(specs=[[{'secondary_y': True}]])

fig_gap_cart.add_trace(
    go.Bar(
        x=monthly_cart['invoice_month'],
        y=monthly_cart['revenue_gap_cart'],
        name='Manque à gagner mensuel (paniers abandonnés)',
        marker_color='#EF553B'
    ), secondary_y=False
)

fig_gap_cart.add_trace(
    go.Scatter(
        x=monthly_cart['invoice_month'],
        y=monthly_cart['cumul_gap_cart'],
        name='Cumul annuel (paniers abandonnés)',
        mode='lines+markers',
        line=dict(color='#636EFA', width=2)
    ), secondary_y=True
)

fig_gap_cart.update_layout(
    title='Manque à gagner mensuel estimé — Paniers abandonnés (GBP)',
    xaxis_title='Mois',
    height=420
)
fig_gap_cart.update_yaxes(title_text='Revenue gap mensuel (paniers abandonnés)', secondary_y=False)
fig_gap_cart.update_yaxes(title_text='Cumul annuel (GBP)', secondary_y=True)
fig_gap_cart.show()

# ── Synthèse des résultats ───────────────────────────────────
total_gap_cart = monthly_cart['revenue_gap_cart'].sum()
total_cart_abandoned = monthly_cart['nb_cart_abandoned'].sum()

print("\n" + "=" * 70)
print("  SYNTHÈSE — PANIERS ABANDONNÉS")
print("=" * 70)
print(f"  Total paniers créés (estimé)      : {monthly_cart['nb_carts_est'].sum():,.0f}")
print(f"  Total commandes finalisées        : {monthly_cart['nb_converted_sessions'].sum():,.0f}")
print(f"  Total PANIERS ABANDONNÉS          : {total_cart_abandoned:,.0f}")
print(f"  Panier moyen converti             : {monthly_cart['avg_basket'].mean():.2f} GBP")
print(f"  Manque à gagner annuel (paniers)  : {total_gap_cart:,.0f} GBP")
print(f"  Soit par mois (moyenne)           : {total_gap_cart/12:,.0f} GBP")
print("=" * 70)


  SYNTHÈSE — PANIERS ABANDONNÉS
  Total paniers créés (estimé)      : 60,892
  Total commandes finalisées        : 17,398
  Total PANIERS ABANDONNÉS          : 43,494
  Panier moyen converti             : 346.97 GBP
  Manque à gagner annuel (paniers)  : 15,226,950 GBP
  Soit par mois (moyenne)           : 1,268,913 GBP


##Q5.Les retards de livraison observés dans le dataset shipping influencent-ils le comportement de retour ou de réachat ?

## 6. KPI 7 — Impact Livraison Tardive

In [ ]:
# ───────────────────────────────────────────────
# KPI 7 — Impact livraison tardive sur RETOUR et RÉACHAT
# ───────────────────────────────────────────────

# ============================================================
# PARTIE 1 : Taux de retour global
# ============================================================

nb_total_orders = df_funnel['nb_invoices'].sum()
nb_returns_total = len(df_returns)

return_rate_global = (
    nb_returns_total / nb_total_orders * 100
)

print("=" * 60)
print("  IMPACT DES RETARDS DE LIVRAISON")
print("=" * 60)
print(f"Taux de retour global          : {return_rate_global:.1f} %")
print(f"Retours totaux / Commandes     : {nb_returns_total:,} / {nb_total_orders:,}")
print()

# ============================================================
# PARTIE 2 : Analyse par statut de livraison
# ============================================================

delivery = (
    df_shipping
    .groupby('delivered_on_time')
    .agg(
        nb_commandes=('order_id', 'count'),
        avg_satisfaction=('customer_rating', 'mean'),
        pct_insatisfait=(
            'customer_rating',
            lambda x: round((x <= 2).mean() * 100, 1)
        ),
        avg_sav_calls=('customer_care_calls', 'mean')
    )
    .reset_index()
)

delivery['statut'] = delivery['delivered_on_time'].map({
    1: 'Livré à temps',
    0: 'En retard'
})

# ============================================================
# PARTIE 3 : Impact estimé sur le taux de retour
# ============================================================

delivery['taux_retour_estime'] = (
    delivery['pct_insatisfait'] / 100
    * return_rate_global
    * 2
).round(1)

# ============================================================
# PARTIE 4 : Impact estimé sur le réachat
# ============================================================

CHURN_RISK = 0.35

delivery['clients_perdus'] = (
    delivery['nb_commandes']
    * delivery['pct_insatisfait'] / 100
    * CHURN_RISK
).round(0)

# AJOUT IMPORTANT :
delivery['taux_clients_perdus'] = (
    delivery['clients_perdus']
    / delivery['nb_commandes']
    * 100
).round(1)

# ============================================================
# AFFICHAGE DES KPI
# ============================================================

print("Résultats par statut de livraison :")
print(
    delivery[
        [
            'statut',
            'nb_commandes',
            'avg_satisfaction',
            'pct_insatisfait',
            'taux_retour_estime',
            'clients_perdus',
            'taux_clients_perdus'
        ]
    ].to_string(index=False)
)

# ============================================================
# VISUALISATION 1
# Satisfaction + Retour estimé
# ============================================================

fig_livraison = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        'Satisfaction moyenne',
        'Taux de retour estimé (%)'
    ]
)

colors = ['#00CC96', '#EF553B']

fig_livraison.add_trace(
    go.Bar(
        x=delivery['statut'],
        y=delivery['avg_satisfaction'],
        marker_color=colors,
        text=delivery['avg_satisfaction'].round(2),
        textposition='outside'
    ),
    row=1,
    col=1
)

fig_livraison.add_trace(
    go.Bar(
        x=delivery['statut'],
        y=delivery['taux_retour_estime'],
        marker_color=colors,
        text=delivery['taux_retour_estime'],
        textposition='outside'
    ),
    row=1,
    col=2
)

fig_livraison.update_layout(
    title='Impact livraison tardive — Satisfaction & retour estimé',
    height=450,
    showlegend=False
)

fig_livraison.update_yaxes(
    title_text='Note moyenne',
    row=1,
    col=1,
    range=[0, 5]
)

fig_livraison.update_yaxes(
    title_text='Taux de retour (%)',
    row=1,
    col=2,
    range=[0, 60]
)

fig_livraison.show()

# ============================================================
# VISUALISATION 2
# Distribution des notes
# ============================================================

fig_boxplot = px.box(
    df_shipping,
    x='delivery_status',
    y='customer_rating',
    color='delivery_status',
    title='Distribution des notes selon le statut de livraison',
    labels={
        'delivery_status': 'Statut livraison',
        'customer_rating': 'Note client'
    },
    color_discrete_map={
        'On Time': '#00CC96',
        'Late': '#EF553B'
    }
)

fig_boxplot.update_layout(
    height=450,
    showlegend=False
)

fig_boxplot.update_yaxes(
    title_text='Note client',
    range=[0.5, 5.5]
)

fig_boxplot.show()

# ============================================================
# PARTIE 5 : Recommandation R3
# ============================================================

nb_retards = delivery.loc[
    delivery['delivered_on_time'] == 0,
    'nb_commandes'
].values[0]

nb_retards_reduit = nb_retards * 0.5

gain_reachat = (
    nb_retards_reduit
    * avg_basket_converted
    * 0.02
)

retours_evites = nb_retards_reduit * 0.01

print()
print("=" * 60)
print("  RECOMMANDATION R3")
print("=" * 60)
print(f"Commandes concernées              : {nb_retards_reduit:,.0f}")
print(f"Gain sur le réachat estimé        : +{gain_reachat:,.0f} GBP")
print(f"Réduction potentielle des retours : ~{retours_evites:,.0f} retours évités")
print("=" * 60)

# ============================================================
# CONCLUSION MÉTIER
# ============================================================

retard = delivery[delivery['delivered_on_time'] == 0].iloc[0]
temps = delivery[delivery['delivered_on_time'] == 1].iloc[0]

ecart_retour = round(
    retard['taux_retour_estime']
    - temps['taux_retour_estime'],
    1
)

ecart_churn = round(
    retard['taux_clients_perdus']
    - temps['taux_clients_perdus'],
    1
)

print()
print("=" * 60)
print("  RÉPONSE À LA QUESTION Q5")
print("=" * 60)

print(f"""
IMPACT SUR LE RETOUR

• Livré à temps :
  - Satisfaction moyenne : {temps['avg_satisfaction']:.2f}
  - Taux de retour estimé : {temps['taux_retour_estime']:.1f} %

• Livraison en retard :
  - Satisfaction moyenne : {retard['avg_satisfaction']:.2f}
  - Taux de retour estimé : {retard['taux_retour_estime']:.1f} %

→ Impact estimé : +{ecart_retour:.1f} point de retour.

IMPACT SUR LE RÉACHAT

• Taux de clients perdus estimé :
  - Livré à temps : {temps['taux_clients_perdus']:.1f} %
  - En retard : {retard['taux_clients_perdus']:.1f} %

→ Impact estimé : +{ecart_churn:.1f} point de perte client.

CONCLUSION

Les retards de livraison sont associés à :
- une satisfaction plus faible,
- davantage d'insatisfaction,
- un taux de retour estimé plus élevé,
- un risque de non-réachat plus important.

La réduction des retards constitue donc un levier direct
d'amélioration de l'expérience client et du chiffre d'affaires.
""")

print("=" * 60)

  IMPACT DES RETARDS DE LIVRAISON
Taux de retour global          : 51.0 %
Retours totaux / Commandes     : 8,872 / 17,398

Résultats par statut de livraison :
       statut  nb_commandes  avg_satisfaction  pct_insatisfait  taux_retour_estime  clients_perdus  taux_clients_perdus
    En retard          4436          2.967989             40.9                41.7           635.0                 14.3
Livré à temps          6563          3.005790             39.4                40.2           905.0                 13.8



  RECOMMANDATION R3
Commandes concernées              : 2,218
Gain sur le réachat estimé        : +15,528 GBP
Réduction potentielle des retours : ~22 retours évités

  RÉPONSE À LA QUESTION Q5

IMPACT SUR LE RETOUR

• Livré à temps :
  - Satisfaction moyenne : 3.01
  - Taux de retour estimé : 40.2 %

• Livraison en retard :
  - Satisfaction moyenne : 2.97
  - Taux de retour estimé : 41.7 %

→ Impact estimé : +1.5 point de retour.

IMPACT SUR LE RÉACHAT

• Taux de clients perdus estimé :
  - Livré à temps : 13.8 %
  - En retard : 14.3 %

→ Impact estimé : +0.5 point de perte client.

CONCLUSION

Les retards de livraison sont associés à :
- une satisfaction plus faible,
- davantage d'insatisfaction,
- un taux de retour estimé plus élevé,
- un risque de non-réachat plus important.

La réduction des retards constitue donc un levier direct
d'amélioration de l'expérience client et du chiffre d'affaires.



##Q6.Y a-t-il des plages horaires ou des jours de la semaine ou la conversion est significativement plus faible?

### 7. Heatmap — Commandes par Jour et Heure

In [ ]:
# ── Récupérer les données horaires depuis fact_transactions ─
df_tx = con.sql('''
    SELECT
        invoice_day_of_week,
        invoice_hour,
        COUNT(DISTINCT invoice_no) AS nb_orders,
        SUM(line_revenue)          AS total_revenue,
        AVG(line_revenue)          AS avg_revenue
    FROM fact_transactions
    GROUP BY invoice_day_of_week, invoice_hour
    ORDER BY invoice_day_of_week, invoice_hour
''').fetchdf()

jours = {0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi',
         4: 'Vendredi', 5: 'Samedi', 6: 'Dimanche'}
df_tx['jour_label'] = df_tx['invoice_day_of_week'].map(jours)

pivot = df_tx.pivot_table(
    index='jour_label',
    columns='invoice_hour',
    values='nb_orders',
    fill_value=0
)
order = ['Lundi','Mardi','Mercredi','Jeudi','Vendredi','Samedi','Dimanche']
pivot = pivot.reindex([j for j in order if j in pivot.index])

fig_heatmap = px.imshow(
    pivot,
    labels={'x': 'Heure', 'y': 'Jour', 'color': 'Nb commandes'},
    title='Heatmap — Commandes par jour × heure (créneaux optimaux de relance)',
    color_continuous_scale='RdYlGn',
    aspect='auto'
)
fig_heatmap.update_layout(height=380)
fig_heatmap.show()

## 8. Synthèse & Recommandations ST1

In [ ]:
!pip uninstall kaleido -y
!pip install kaleido==0.2.1
!pip install plotly>=5.24.0

# Vérifier l'installation
import kaleido
print("Kaleido version:", kaleido.__version__)

Found existing installation: kaleido 0.2.1
Uninstalling kaleido-0.2.1:
  Successfully uninstalled kaleido-0.2.1
  Using cached kaleido-0.2.1-py2.py3-none-manylinux1_x86_64.whl.metadata (15 kB)
Using cached kaleido-0.2.1-py2.py3-none-manylinux1_x86_64.whl (79.9 MB)


ERROR: Operation cancelled by user
^C
Kaleido version: 0.2.1


In [ ]:
# ── Installer Kaleido (à exécuter UNE SEULE FOIS) ──
!pip install -U kaleido -q

# ── Export de tous les graphiques dans un dossier ─────────
import os

OUTPUT = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Notebooks/Graphiques_ST1'
os.makedirs(OUTPUT, exist_ok=True)

# Vérifiez d'abord quelles figures existent
figures_disponibles = {}
for nom in ['fig_funnel', 'fig_gap', 'fig_map', 'fig_pays', 'fig_prix',
            'fig_paniers', 'fig_heatmap', 'fig_livraison', 'fig_boxplot',
            'fig_mensuel', 'fig_scatter']:
    try:
        fig = eval(nom)
        figures_disponibles[nom] = fig
        print(f"✓ {nom} trouvée")
    except NameError:
        print(f"✗ {nom} non trouvée")

# Mapper noms de fichiers → figures disponibles
figures = {
    '01_funnel_conversion'   : figures_disponibles.get('fig_funnel'),
    '02_manque_a_gagner'     : figures_disponibles.get('fig_gap'),
    '03_carte_pays'          : figures_disponibles.get('fig_map'),
    '04_top_pays_hors_uk'    : figures_disponibles.get('fig_pays'),
    '05_abandon_par_prix'    : figures_disponibles.get('fig_prix'),
    '06_distribution_paniers': figures_disponibles.get('fig_paniers'),
    '07_heatmap_jour_heure'  : figures_disponibles.get('fig_heatmap'),
    '08_impact_livraison'    : figures_disponibles.get('fig_livraison'),
    '09_boxplot_livraison'   : figures_disponibles.get('fig_boxplot'),
    '10_evolution_mensuelle' : figures_disponibles.get('fig_mensuel'),
    '11_scatter_categories'  : figures_disponibles.get('fig_scatter'),
}

# Exporter seulement les figures qui existent
compteur = 0
for nom, fig in figures.items():
    if fig is not None:
        try:
            fig.write_image(f'{OUTPUT}/{nom}.png')
            print(f"✓ {nom}.png exporté")
            compteur += 1
        except Exception as e:
            print(f"✗ {nom}.png - Erreur : {e}")
    else:
        print(f"✗ {nom} ignoré (figure absente)")

print(f'\n {compteur} graphiques exportés dans :')
print(f'   {OUTPUT}')

In [ ]:
# ── Tableau de synthèse des KPIS (CORRIGÉ) ────────────────────

# Utiliser monthly_cart c'est la variable pour les paniers abandonnés
total_gap_cart = monthly_cart['revenue_gap_cart'].sum()
best_month = monthly_cart.loc[monthly_cart['nb_converted_sessions'].idxmax(), 'invoice_month']
worst_month = monthly_cart.loc[monthly_cart['nb_converted_sessions'].idxmin(), 'invoice_month']


best_delivery = delivery.loc[delivery['avg_satisfaction'].idxmax(), 'statut']

print('=' * 60)
print('  SYNTHÈSE ST1 — OBSERVATOIRE CONVERSION')
print('=' * 60)
print(f'  Taux de conversion global (benchmark) : 3.0 %')
print(f'  Taux d\'abandon panier (estimé)        : 71.4 %')
print(f'  Panier moyen converti                 : {avg_basket_converted:.2f} GBP')
print(f'  Manque à gagner annuel (paniers)      : {total_gap_cart:,.0f} GBP')
print(f'  Meilleur mois (volume commandes)      : {best_month}')
print(f'  Mois le plus faible                   : {worst_month}')
print(f'  Meilleur statut livraison             : {best_delivery}')
print()
print('  RECOMMANDATIONS (CDC §3.5) :')
print()
print('  R1 — Relance email sur paniers abandonnés (>100 GBP)')
print(f'       Impact estimé (récupération 5%)   : +{total_gap_cart*0.05:,.0f} GBP/an')
print()
print('  R2 — Optimiser les créneaux de relance (heatmap)')
print('       Cibler : mardi, mercredi, jeudi de 10h à 18h')
print()
print('  R3 — Réduire les retards de livraison de 50%')
print('       → Gain estimé : +15 528 GBP + ~22 retours évités')
print('=' * 60)

con.close()
print('\nConnexion DuckDB fermée ')